# Building an accessor - Met Office UKV data
In this tutorial we will demonstrate how to build a simple data accessor for pyearthtools, which is a key building block in a PyEarthTools pipeline. For this example we will use a small sample of Forecast Analysis data from the Met Office , which is available through AWS. Although one can 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pathlib
import functools
import datetime

In [ ]:
import matplotlib.pyplot
import cartopy
import numpy

In [ ]:
import xarray

In [ ]:
import h5py

### Exploring data
Let's start by looking at the data we are going to load.  A number of [Met Office Datasets](https://www.metoffice.gov.uk/services/data/external-data-channels) are available through third-party sources such as AWS and Snowflake. In this example we will look at loading some of UK high-resolution analysis data from AWS.

Links:
* [Met Office Data on Thiurd-party platforms](Links:https://www.metoffice.gov.uk/services/data/external-data-channels)
* [Met Office UKV data documentation](https://www.metoffice.gov.uk/api/assets/file/ukv-2km-ps47-asdi-pdf-updatespdf?prefix=assets)
* [UKV on AWS Sustainable Data Initiative](https://registry.opendata.aws/met-office-uk-deterministic/)
* [Contents of AWS S3 bucket for MO UKV data](https://met-office-atmospheric-model-data.s3.eu-west-2.amazonaws.com/index.html#uk-deterministic-2km/)

In [ ]:
open_args = {
    'engine':"h5netcdf", #
    'storage_options': {"anon": True},
}

In [ ]:
mf_args = {'concat_dim' : 'time',
           'combine' :'nested'
          }

In [ ]:
ukv_humidity_sample_ds = xarray.open_dataset(
    's3://met-office-atmospheric-model-data/uk-deterministic-2km/20260601T1200Z/20260601T1200Z-PT0000H00M-relative_humidity_on_pressure_levels.nc',
    ** open_args,
)

In [ ]:
ukv_humidity_sample_ds

In [ ]:
ukv_humidity_sample_ds.pressure

In [ ]:
data_projection = cartopy.crs.LambertAzimuthalEqualArea(central_latitude=54.9, central_longitude=-2.5, false_easting=0.0, false_northing=0.0)

In [ ]:
fig1 = matplotlib.pyplot.figure()
ax1 = fig1.add_subplot(1,1,1,projection=cartopy.crs.OSGB() )
ukv_humidity_sample_ds.sel({'pressure':85000})['relative_humidity'].plot.contourf(
    ax=ax1,
    transform=data_projection,
)
ax1.coastlines()

### Create a larger dataset from multiple files

In [ ]:
dt_list = [datetime.datetime(2026,6,1,0,0) + ix1 * datetime.timedelta(hours=12) for ix1 in range(20)]

In [ ]:
var_list = [
    'temperature_on_pressure_levels',
    'relative_humidity_on_pressure_levels',
    'wind_speed_on_pressure_levels',
    'wind_direction_on_pressure_levels',
]

In [ ]:
vt_template= '{dt.year:04d}{dt.month:02d}{dt.day:02d}T{dt.hour:02d}{dt.minute:02d}Z'

In [ ]:
ukv_fname_template = '{vt_str}-PT0000H00M-{var_name}'

In [ ]:
ukv_path_dict = {current_var: [
    's3://met-office-atmospheric-model-data/uk-deterministic-2km/{vt_str}/{vt_str}-PT0000H00M-{var_name}.nc'.format(
        vt_str=vt_template.format(dt=current_dt),
        fname=ukv_fname_template.format(vt_str=vt_template.format(dt=current_dt), var_name=current_var),
    )
    # for current_dt in dt_list for current_var in ['temperature_on_pressure_levels']
    for current_dt in dt_list 
] for current_var in var_list }

In [ ]:
ukv_path_dict

In [ ]:
ukv_sample_ds = xarray.merge([xarray.open_mfdataset(var_paths, ** (open_args | mf_args) ) 
                      for current_var,var_paths in  ukv_path_dict.items()])

In [ ]:
ukv_sample_ds

In [ ]:
select_time = datetime.datetime(2026,6,7,12,0)

In [ ]:
ukv_sample_ds.time

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(18,6))
# plot temperature at 850hpa pressure level
ax1 = fig1.add_subplot(1,3,1,projection=cartopy.crs.OSGB() )
ukv_sample_ds.sel({'pressure':85000, 'time': select_time})['air_temperature'].plot.contourf(
    ax=ax1,
    transform=data_projection,
)
ax1.coastlines()


ax1 = fig1.add_subplot(1,3,2,projection=cartopy.crs.OSGB() )

ukv_sample_ds.sel({'pressure':85000, 'time': select_time})['relative_humidity'].plot.contourf(
    ax=ax1,
    transform=data_projection,
)
ax1.coastlines()

ax1 = fig1.add_subplot(1,3,3,projection=cartopy.crs.OSGB() )
ukv_sample_ds.sel({'pressure':85000, 'time': select_time})['wind_speed'].plot.contourf(
    ax=ax1,
    transform=data_projection,
)
ax1.coastlines()


### Download the data

this could be done through a DownloadIndex in future. This will be done manually for now in this tutorial to demonstrate

In [ ]:
import tempfile

In [ ]:
# change this path to a suitable local directory for which you have write permission
local_dir_obj = tempfile.TemporaryDirectory()
local_dir = pathlib.Path(local_dir_obj.name)
local_dir

In [ ]:
with tempfile.TemporaryDirectory()as td1:
    print(type(td1))

In [ ]:
pathlib.Path(ukv_path_dict[var_list[0]][1]).suffix

In [ ]:
local_path_dict = {}
for var_name, path_list in ukv_path_dict.items():
    for path1 in path_list:
        print(var_name)
        out_path = local_dir / (pathlib.Path(path1).stem + pathlib.Path(path1).suffix)
        xarray.open_dataset(path1, **open_args).to_netcdf(out_path)
        local_path_dict[var_name]+= [local_dir / (pathlib.Path(path1).stem + pathlib.Path(path1).suffix)]

In [ ]:
[f1 for f1 in local_dir.iterdir()]

### Create PyEarthTools Accessor

In [ ]:
import pyearthtools

In [ ]:
from pyearthtools.data import Petdt
from pyearthtools.data.indexes import ArchiveIndex
from pyearthtools.data.transforms import Transform, TransformCollection
from pyearthtools.data.archive import register_archive

In [ ]:
class MO_UKV(ArchiveIndex):
    """
    """
    MO_UKV_AWS_ROOT_PATH = 's3://met-office-atmospheric-model-data/uk-deterministic-2km/'
    MO_UKV_AWS_DIR_TEMPLATE = MO_UKV_AWS_ROOT_PATH + '{vt_str}'
    MO_UKV_FNAME_TEMPLATE = '{vt_str}-PT0000H00M-{var_name}.nc'
    
    def __init__(
        self,
        root_dir : pathlib.Path,
        variables: list[str] | str,
        *,
        transforms: Transform | TransformCollection | None = None,
    ):
        """
        Init function for Merra2 accessor base class.
        """
        self._root_dir = root_dir
        self._variables = variables
        super_transforms = TransformCollection([pyearthtools.data.transforms.variables.Trim(self._variables),]) + transforms
        
        # call the base class
        super().__init__(
            transforms=super_transforms,
        )
        self.record_initialisation()

    def filesystem(
        self,
        querytime: str | Petdt,
    ) -> pathlib.Path | dict[str, str | pathlib.Path]:
        

        querytime = Petdt(querytime)
        paths = []
        
        
        for var_name in self._variables:
            try:
                current_fname = MO_UKV.MO_UKV_FNAME_TEMPLATE.format(vt_str=vt_template.format(dt=querytime), var_name=var_name)
                if self._root_dir is None:
                    current_dir = MO_UKV.MO_UKV_AWS_DIR_TEMPLATE.format(vt_str=vt_template.format(dt=querytime))
                    current_path = f'{current_dir}/{current_fname}'
                else:
                    current_dir = self._root_dir
                    current_path = current_dir / current_fname
                
                paths += [ current_path ]
            except KeyError:
                print(f'Non data for var {var_name}')
        
        print(paths)
        return paths
        
    def __desc__(self):
        return {
            "singleline": "Met Office UKV Forecast Analysis ",
            "range": "June 2026",
            "Documentation": "https://registry.opendata.aws/met-office-uk-deterministic/",
        }

    

In [ ]:
ukv_accessor = MO_UKV(local_dir, var_list)

In [ ]:
ukv_accessor['2026-06-01 12:00']

### Creating an accessor for Zarr data
We could instead have saved the data in a format that optimised for object stores as found on cloud platforms, such as zarr. Different formats will give better performance on different platforms based on the specifics of the storage hardward and configuration, so although zarr is efficient in many cases, netcdf or another may be more appropriate in other cases.

In [ ]:
zarr_dir = local_dir / 'zarr_data'
zarr_dir.mkdir()

In [ ]:
xarray.merge([xarray.open_mfdataset(var_paths, ** ( mf_args) ) 
                      for current_var,var_paths in  local_path_dict.items()]).to_zarr(zarr_dir)

In [ ]:
zarr_accessor = pyearthtools.data.archive.ZarrTimeIndex(
    zarr_dir,
    variables=var_list,
    remote=False,
)

In [1]:
## Building a remote accssor
We could aloso just build an accessor which directly loads the data from the AWS S3 bucket. This requires some extra argument be provided to thhe data index, as we used with the `xarray.open_dataset` functionm when we called it directly.

SyntaxError: invalid syntax (890097016.py, line 2)